# Mission 01: A Taxi Batch Pipeline at Scale

The city's mobility team receives the full monthly taxi export (about 3 million trips per month) and wants two
"gold" tables every morning: revenue per borough per day, and demand per zone and hour. Build the batch pipeline in
Spark, make it correct, and tune it with measurements instead of guesses.

**Lessons used:** 01 (architecture), 02 (plans), 03 (partitioning and the shuffle), 04 (joins), 10 (tuning).

**Data:** NYC yellow taxi trips, January and February 2024, and the taxi zone lookup table.

**Services:** none (Spark runs locally in the notebook).

In [ ]:
import shutil
import sys
import tempfile
import time
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets                                   # course helper: pinned datasets

warnings.filterwarnings("ignore", message=".*pandas >= 3.0.0.*")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
WORK = Path(tempfile.gettempdir()) / "fzth-m13" / "mission01"
shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir(parents=True)
RAW = [str(datasets.taxi("2024-01")), str(datasets.taxi("2024-02"))]
ZONES = str(datasets.fetch("taxi_zone_lookup.csv"))

spark = (SparkSession.builder.master("local[4]").appName("m13-mission01")
         .config("spark.driver.memory", "1500m").config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl, "| work dir:", WORK)

## Tasks

1. **Clean.** Read both months. Keep trips with a pickup inside January–February 2024, `trip_distance` in (0, 100],
   `total_amount` in [0, 1000], and a known pickup zone (1–263). Report how many rows each rule removes.
2. **Enrich.** Join the zone lookup (borough, zone name) and add `pickup_date` and `pickup_hour`. Show in the plan
   which join strategy Spark chose and why.
3. **Gold tables.** Build `daily_borough_revenue` (date, borough, trips, revenue, average tip %) and
   `zone_hour_demand` (zone, hour of day, weekday/weekend, average trips per day). Write the first one as Parquet,
   partitioned by month; report the number of files and bytes.
4. **Tune.** Time the gold build with (a) 200 shuffle partitions and AQE off, (b) 8 shuffle partitions, AQE off,
   (c) AQE on. Report time, number of tasks and shuffle bytes from the Spark REST API, and choose a setting.
5. **Verify.** Recompute `daily_borough_revenue` with DuckDB SQL directly on the Parquet files with the same rules.
   The two results must be equal (trips exactly, revenue within one cent).
6. **Explain.** Five sentences for the mobility team: what the pipeline guarantees, which setting you chose and why,
   and what would change on a real cluster with 5 years of data.

In [ ]:
# Tasks 1-6
...

## Submission Checklist

- [ ] Every cleaning rule has a row count; nothing is dropped silently.
- [ ] The join strategy is shown in a plan, not assumed.
- [ ] The tuning table comes from measurements (time, tasks, shuffle bytes).
- [ ] Spark and DuckDB give the same gold table.

A solution is in `01-Taxi-Batch-Pipeline-at-Scale-solution.ipynb`. Try the mission first.